In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

Import dataset

In [ ]:

df = pd.read_csv(r"data\raw\Missile Project 1 csv - deepseek_csv_20260306_df9f2e.csv")
print(f"Loaded {len(df)} rows")
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'data\\raw\\Missile Project 1 csv - deepseek_csv_20260306_df9f2e.csv'

Remove Duplicate

In [233]:
df = df.drop_duplicates(subset=['Missile_ID','Missile_Name','Alternate_Name','Country_of_Origin','Country_of_Possession','Missile_Class','Missile_Type','Length_m','Launch_Weight_kg','Warhead_Type','Range_min_km','Range_max_km','Year_Introduced','Operational_Status','Launch_Basing','Diameter_m','Payload_kg','Propulsion_Type','Notes'])
print(f"After removing duplicates: {len(df)} rows")

After removing duplicates: 369 rows


Structure data and fix Country_of_Possession duplicate

In [ ]:
import pandas as pd

# Load the DataFrame
df = pd.read_csv('/content/Missile Project 1 csv - deepseek_csv_20260306_df9f2e.csv')

# Remove duplicates (as per cell XkO6SIs68dDi)
df = df.drop_duplicates(subset=['Missile_ID','Missile_Name','Alternate_Name','Country_of_Origin','Country_of_Possession','Missile_Class','Missile_Type','Length_m','Launch_Weight_kg','Warhead_Type','Range_min_km','Range_max_km','Year_Introduced','Operational_Status','Launch_Basing','Diameter_m','Payload_kg','Propulsion_Type','Notes'])

# Remove extra spaces from column names
df.columns = df.columns.str.strip()

# Clean text columns
text_columns = [
    'Missile_Name',
    'Alternate_Name',
    'Country_of_Origin',
    'Country_of_Possession',
    'Missile_Class',
    'Missile_Type',
    'Warhead_Type',
    'Operational_Status',
    'Launch_Basing',
    'Propulsion_Type',
    'Notes'
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# Aggregation rules
agg = {}

for col in df.columns:
    if col == "Country_of_Possession":
        # Merge all unique countries into one cell
        agg[col] = lambda x: ", ".join(sorted(set(i for i in x if i)))
    elif col != "Missile_ID":
        # Keep the first value for every other column
        agg[col] = "first"

# Group by Missile_ID
df_clean = df.groupby("Missile_ID", as_index=False).agg(agg)

# Save cleaned dataset
df_clean.to_csv("missiles_clean.csv", index=False)

print(df_clean.head())
print(f"\nOriginal Rows: {len(df)}")
print(f"Clean Rows: {len(df_clean)}")

In [235]:
# Dictionary for standardizing country names
country_mapping = {
    'United States': 'United States',
    'USA (United States)': 'United States',
    'PRC': 'China',
    'China (PRC)': 'China',
    'Russian Federation': 'Russia',
    'Russian': 'Russia'
}

# Replace country names in both columns
df['Country_of_Origin'] = df['Country_of_Origin'].replace(country_mapping)
df['Country_of_Possession'] = df['Country_of_Possession'].replace(country_mapping)

# Check the updated counts
print("Country of Origin:")
print(df['Country_of_Origin'].value_counts())

print("\nCountry of Possession:")
print(df['Country_of_Possession'].value_counts())

Country of Origin:
Country_of_Origin
China              65
USA                52
Iran               35
North Korea        32
Russia             29
Pakistan           27
Israel             25
South Korea        20
Taiwan             17
France             17
UK                 15
India              12
Israel/US           5
UK/France           3
France/Europe       3
US                  3
Israel/India        1
France/UK           1
Russia/India        1
USA (Kongsberg)     1
US/UK               1
UK/Italy            1
Norway/UK           1
UK/Europe           1
UK/Sweden           1
Name: count, dtype: int64

Country of Possession:
Country_of_Possession
China                     60
USA                       50
Iran                      35
Pakistan                  33
North Korea               32
Israel                    30
Russia                    29
UK                        26
South Korea               20
Taiwan                    20
France                    15
India                 

Standardize missile types

In [236]:
type_mapping = {
    'SRBM': 'SRBM',
    'Short Range Ballistic Missile': 'SRBM',
    'MRBM': 'MRBM',
    'Medium Range Ballistic Missile': 'MRBM',
    'ICBM': 'ICBM',
    'Intercontinental Ballistic Missile': 'ICBM',
    'IRBM': 'IRBM',
    'Intermediate Range Ballistic Missile': 'IRBM',
    'Cruise': 'Cruise Missile',
    'SLBM': 'SLBM',
    'Submarine Launched Ballistic Missile': 'SLBM'
}
df['Missile_Type'] = df['Missile_Type'].replace(type_mapping)

Clean range values

In [237]:
# List of range columns
range_columns = ['Range_min_km', 'Range_max_km']

for col in range_columns:
    # Convert to string
    df[col] = df[col].astype(str)

    # Remove unwanted text/characters
    df[col] = df[col].str.replace('km', '', case=False, regex=False)
    df[col] = df[col].str.replace(',', '', regex=False)
    df[col] = df[col].str.replace(' ', '', regex=False)

    # Convert to numeric
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove rows with invalid ranges
df = df[
    (df['Range_min_km'] >= 0) &
    (df['Range_max_km'] > 0) &
    (df['Range_max_km'] < 20000) &
    (df['Range_max_km'] >= df['Range_min_km'])
]

Clean years

In [238]:
# Convert Year_Introduced to string
df['Year_Introduced'] = df['Year_Introduced'].astype(str)

# Extract only the 4-digit year (e.g., "1998", "2005")
df['Year_Introduced'] = df['Year_Introduced'].str.extract(r'(\d{4})')[0]

# Convert to numeric
df['Year_Introduced'] = pd.to_numeric(df['Year_Introduced'], errors='coerce')

# Keep only realistic years
df = df[
    (df['Year_Introduced'] >= 1900) &
    (df['Year_Introduced'] <= 2025)
]

Clean status

In [239]:
# Remove extra spaces
df['Operational_Status'] = df['Operational_Status'].astype(str).str.strip()

# Standardize values
status_mapping = {
    'Active': 'Active',
    'Operational': 'Active',
    'In Service': 'Active',

    'Retired': 'Retired',
    'Decommissioned': 'Retired',

    'Development': 'Development',
    'Testing': 'Development',
    'Experimental': 'Development'
}

df['Operational_Status'] = df['Operational_Status'].replace(status_mapping)

Handle missing values

In [240]:
# Fill missing Operational_Status with 'Unknown'
df['Operational_Status'] = df['Operational_Status'].fillna('Unknown')

# Fill missing Missile_Type with 'Unknown'
df['Missile_Type'] = df['Missile_Type'].fillna('Unknown')

# Drop rows where Missile_Name or Country_of_Origin is missing
df = df.dropna(subset=['Missile_Name', 'Country_of_Origin'])

Add derived columns

In [241]:

# Classify missiles by maximum range
def classify_by_range(range_km):
    if pd.isna(range_km):
        return 'Unknown'
    elif range_km < 300:
        return 'Short Range'
    elif range_km < 1000:
        return 'Medium Range'
    elif range_km < 3500:
        return 'Intermediate Range'
    elif range_km < 5500:
        return 'Long Range'
    else:
        return 'Intercontinental Range'

# Create Range_Category column
df['Range_Category'] = df['Range_max_km'].apply(classify_by_range)

# Create Decade column
df['Decade'] = (df['Year_Introduced'] // 10 * 10).astype('Int64')

Sort

In [242]:
df = df.sort_values(
    ['Country_of_Origin', 'Country_of_Possession', 'Missile_Name']
).reset_index(drop=True)

Preview cleaned data

In [ ]:
print("\n" + "="*60)
print("🚀 MISSILE DATASET SUMMARY")
print("="*60)

print(f"Total Missiles              : {len(df)}")
print(f"Origin Countries           : {df['Country_of_Origin'].nunique()}")
print(f"Possession Countries       : {df['Country_of_Possession'].nunique()}")
print(f"Missile Classes            : {df['Missile_Class'].nunique()}")
print(f"Missile Types              : {df['Missile_Type'].nunique()}")
print(f"Warhead Types              : {df['Warhead_Type'].nunique()}")
print(f"Launch Platforms           : {df['Launch_Basing'].nunique()}")
print(f"Propulsion Types           : {df['Propulsion_Type'].nunique()}")

print(f"Range (Min)                : {df['Range_min_km'].min():.0f} km")
print(f"Range (Max)                : {df['Range_max_km'].max():.0f} km")

print(f"Year Introduced            : {df['Year_Introduced'].min():.0f} - {df['Year_Introduced'].max():.0f}")

In [244]:

report = f"""
========================================
MISSILE DATA CLEANING REPORT
========================================
File Processed           : missile_data.csv

Total Missiles           : {len(df)}

Origin Countries         : {df['Country_of_Origin'].nunique()}
Possession Countries     : {df['Country_of_Possession'].nunique()}

Missile Classes          : {df['Missile_Class'].nunique()}
Missile Types            : {df['Missile_Type'].nunique()}

Minimum Range            : {df['Range_min_km'].min():.0f} km
Maximum Range            : {df['Range_max_km'].max():.0f} km

Year Introduced          : {df['Year_Introduced'].min():.0f} - {df['Year_Introduced'].max():.0f}

Range Category Breakdown
------------------------
{df['Range_Category'].value_counts().to_string()}

Operational Status Breakdown
----------------------------
{df['Operational_Status'].value_counts().to_string()}

Missing Values
--------------
{df.isnull().sum().to_string()}

========================================
"""

print(report)


MISSILE DATA CLEANING REPORT
File Processed           : missile_data.csv

Total Missiles           : 264

Origin Countries         : 23
Possession Countries     : 14

Missile Classes          : 76
Missile Types            : 9

Minimum Range            : 0 km
Maximum Range            : 18000 km

Year Introduced          : 1953 - 2025

Range Category Breakdown
------------------------
Range_Category
Short Range               112
Medium Range               66
Intermediate Range         49
Intercontinental Range     23
Long Range                 14

Operational Status Breakdown
----------------------------
Operational_Status
Active                 217
Obsolete                30
Retired                  7
In development           5
Operational Limited      2
Cancelled                1
Being replaced           1
Phased out               1

Missing Values
--------------
Missile_ID                 0
Missile_Name               0
Alternate_Name             0
Country_of_Origin          0
Country